# Baseline 3 — detector trained with copy-paste augmentation

Trains Faster R-CNN on NODE21 plus pasted synthetic positives, then evaluates with the
same FROC code as Baseline 1.

## The recipe and split are no longer unknown

Recovered 2026-09-12 from `train_baseline.py`, `algoverse_dataset.py` and `splits.csv`.
This notebook now matches them exactly rather than guessing, so **Baseline 1's existing
checkpoint is a valid comparator** and only one arm has to be trained — 6 to 10 GPU-hours,
not 12 to 20.

Four of the previous assumptions in this notebook were wrong:

| | previously assumed | actual |
|---|---|---|
| backbone weights | `None` — random init | **`"DEFAULT"` — COCO-pretrained** |
| batch_size | 4 | **2** |
| seed | 0 | **42** |
| train-time aug | the paste code only | `ToTensor` + `RandomHorizontalFlip(0.5)` |

`weights=None` versus `"DEFAULT"` is training from scratch versus fine-tuning COCO. That
alone would have made any FROC comparison meaningless.

## Preprocessing is the upstream one, not ours

`NoduleDataset` max-normalises and does nothing else — no CLAHE, no percentile clip, no
crop, no resize, boxes in native pixels. Rather than reimplement it and risk drift, §5
unzips `node21_detection_baseline.zip` and imports the real `training_utils` and
`algoverse_dataset`, exactly as the original run did.

> **Still true, and still a limitation.** NODE21 ships no patient identifiers, so the split
> is image-level and two studies of one patient can sit on opposite sides. Recovering the
> original split makes the comparison *internally consistent*; it does not fix this.

> **Also still true.** 8 of the 12 grid source chests were in this detector's training set
> (`config.json` in Dhruv's predictor bundle). That confounds the synthetic arm, not this
> one, but it belongs in the same Limitations paragraph. See FINDINGS §2.2.

## 1 · Mount, paths, checkpoint plumbing

In [ ]:
# ===========================================================================
# CANONICAL DRIVE PATHS
# Mirrored from src/paths.py and notebooks/CONFIG_CELL.md. Mapped from Drive
# 2026-09-12 with the Drive connector. Change all three together.
# Full layout, folder ids and the old->new table: DRIVE_LAYOUT.md
# ===========================================================================
!pip -q install SimpleITK
import os, json, glob, time, shutil, subprocess, random
from pathlib import Path
from google.colab import drive

# --- mount -----------------------------------------------------------------
# ismount(), not isdir(). A plain local directory under an unmounted
# /content/drive is also a dir, and creating one blocks the mount and makes
# Drive look empty -- that happened once and looked like a wiped Drive.
if not os.path.ismount('/content/drive'):
    if Path('/content/drive').exists():
        os.system('fusermount -u /content/drive 2>/dev/null')
        shutil.rmtree('/content/drive', ignore_errors=True)
    drive.mount('/content/drive')
assert Path('/content/drive/MyDrive').is_dir(), 'mount failed'

# --- resolve the root ------------------------------------------------------
# MyDrive/Algoverse is a SHORTCUT to the shared folder Feliciano_Algoverse.
# One folder, not two; the FUSE mount resolves it as a directory.
# Test for the 01_data marker, never for the root itself: an unresolved
# shortcut and a stale empty directory both "exist", and that is exactly how
# a stray empty results/ tree got created on 2026-09-12.
ROOT = None
for _c in ['/content/drive/MyDrive/Algoverse',
           '/content/drive/MyDrive/Feliciano_Algoverse',
           '/content/drive/Shareddrives/Feliciano_Algoverse']:
    if (Path(_c)/'01_data').is_dir():
        ROOT = Path(_c); break
assert ROOT is not None, (
    'Algoverse root not found. Tried MyDrive/Algoverse, '
    'MyDrive/Feliciano_Algoverse, Shareddrives/Feliciano_Algoverse.\n'
    f'MyDrive top level: '
    f'{sorted(p.name for p in Path("/content/drive/MyDrive").iterdir())[:20]}\n'
    'MyDrive/Algoverse is a shortcut to the shared Feliciano_Algoverse. If it '
    'is gone: Drive -> Shared with me -> right-click Feliciano_Algoverse -> '
    'Add shortcut to Drive -> My Drive.')

# --- layout ----------------------------------------------------------------
SOURCE    = ROOT/'01_data'/'00_source'           # node21, chexpert, mimic_cxr
NODE21    = SOURCE/'node21'
MHA_SRC   = NODE21/'images'                      # 4,882 .mha
ANN_CSV   = NODE21/'metadata.csv'                # 5,224 rows, 1,476 label==1
GRID      = ROOT/'01_data'/'01_grid'
GRID_CSV  = GRID/'grid_v5.csv'                   # 231,145 bytes if it is the right one
RUNS      = GRID/'_runs'                         # generation checkpoint zips, not data
EMB_DIR   = ROOT/'01_data'/'02_embeddings'
CPASTE    = ROOT/'01_data'/'03_copypaste'
MODELS    = ROOT/'02_results'/'00_models'
CKPT      = MODELS/'baseline1_checkpoint.pth'    # .pth -- the old .pt path is dead
FIGS      = ROOT/'02_results'/'01_figures'
B1_DIR    = ROOT/'02_results'/'02_baselines'
PRED_DIR  = ROOT/'02_results'/'03_predictor'
B3_DIR    = ROOT/'02_results'/'04_baseline3'

print(f'root: {ROOT}')

OUT  = Path('/content/baseline3')
DEST = B3_DIR
for d in [OUT/'aug', OUT/'ckpt', OUT/'dets']:  d.mkdir(parents=True, exist_ok=True)
for d in [DEST/'ckpt', DEST/'dets']:           d.mkdir(parents=True, exist_ok=True)

# --- verify inputs ---------------------------------------------------------
for _p in (MHA_SRC, ANN_CSV):
    assert _p.exists(), (
        f'{_p} missing.\n  its parent holds: '
        f'{sorted(q.name for q in _p.parent.iterdir())[:12] if _p.parent.is_dir() else "parent does not exist either"}'
        '\n  see the old->new table in DRIVE_LAYOUT.md')
print('inputs ok')

# --- stray folders from the pre-reorganisation layout ----------------------
_stray = [ROOT/'results', ROOT/'data', ROOT/'artifacts', ROOT/'grid_v5b_runs',
          Path('/content/drive/MyDrive/grid_v5_runs'),
          Path('/content/drive/MyDrive/grid_v4_runs'),
          Path('/content/drive/MyDrive/Teammates')]
_hits = [(p, len(list(p.iterdir()))) for p in _stray if p.is_dir()]
if _hits:
    print('\nstray folders from the old layout (DRIVE_LAYOUT.md lists what to do):')
    for p, n in _hits:
        print(f'  {p}   ({"empty, safe to delete" if n == 0 else str(n) + " items -- CHECK"})')


def to_drive(sub):
    n = 0
    for f in (OUT/sub).iterdir():
        if not (DEST/sub/f.name).exists():
            shutil.copy(f, DEST/sub/f.name); n += 1
    return n

def from_drive(sub):
    n = 0
    for f in (DEST/sub).iterdir():
        if not (OUT/sub/f.name).exists():
            shutil.copy(f, OUT/sub/f.name); n += 1
    return n

print(f'\nrestored {from_drive("ckpt")} checkpoints, {from_drive("dets")} detection files')
print(f'writing to {DEST}')
print(f"Baseline 1 FROC for comparison will be read from {B1_DIR/'table-froc.csv'}")


## 2 · Configuration

`EPOCHS`, `LR` and the rest are the numbers that must match Baseline 1. They are grouped
here so they can be replaced in one place once the original recipe is found.

In [ ]:
# ===========================================================================
# RECOVERED from train_baseline.py -- do not "improve" these. They define
# comparability with baseline1_checkpoint.pth, which is epoch_5 of this recipe.
# ===========================================================================
EPOCHS       = 5
BATCH        = 2            # was 4 in the guessed version
LR           = 0.005
MOMENTUM     = 0.9
WEIGHT_DECAY = 0.0005
LR_STEP      = 3
LR_GAMMA     = 0.1
SEED         = 42           # was 0 in the guessed version
PRETRAINED   = 'DEFAULT'    # COCO weights. was None -- random init. critical.
WORKERS      = 2

SCORE_MIN    = 0.05         # operating point for FROC, not a filter

# Canvas geometry for the pasted images. Real images go in at NATIVE resolution
# (see section 5); SIZE applies only to the augmented PNGs we synthesise.
SIZE         = 512
DET_SIZE     = 800

AUG_PER_CHEST = 2           # synthetic positives per clean background
N_AUG_CHESTS  = 600         # clean chests to paste into

CHECKPOINT_EVERY_EPOCH = True     # non-negotiable at 6-10 h against a 12 h cap

import random, numpy as np, torch
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
assert DEV == 'cuda', 'training on CPU is not viable here'
print(torch.__version__, torch.version.cuda, torch.cuda.get_device_name(0))
print(f'recipe: epochs={EPOCHS} batch={BATCH} lr={LR} seed={SEED} weights={PRETRAINED}')

## 3 · Split — the recovered one

`splits.csv` is the split Baseline 1 trained on, generated deterministically by
`build_splits.py` from `metadata.csv`. 77,012 bytes; identical copies exist in Maia's Drive
folder and as `baseline1_splits.csv` in Dhruv's predictor bundle.

**This notebook no longer invents a split.** The previous version wrote its own deterministic
one and warned that comparisons against Baseline 1 were invalid. That warning is now
obsolete — but only if this cell finds the real file, so it refuses to fall back.

In [ ]:
import pandas as pd

SPLIT_CSV = None
for _c in [SOURCE/'baseline1_splits.csv', SOURCE/'splits.csv',
           NODE21/'baseline1_splits.csv', NODE21/'splits.csv']:
    if _c.exists():
        SPLIT_CSV = _c; break
if SPLIT_CSV is None:
    _h = sorted(glob.glob('/content/drive/MyDrive/**/baseline1_splits.csv', recursive=True)
                + glob.glob('/content/drive/MyDrive/**/splits.csv', recursive=True)
                + glob.glob('/content/*splits.csv'))
    assert _h, (
        'splits.csv NOT FOUND, and this notebook will not invent one -- a fabricated split '
        'makes the comparison against Baseline 1 invalid, which is the whole problem this '
        'recovery solved.\n'
        f'Copy it to {SOURCE}/baseline1_splits.csv. It is 77,012 bytes, in Maia\'s Drive '
        'folder and at Algoverse_Predictor_Code_Outputs/data/splits/baseline1_splits.csv.')
    SPLIT_CSV = Path(_h[0]); print(f'found by glob: {SPLIT_CSV}')

SPL = pd.read_csv(SPLIT_CSV)
assert {'img_name', 'split'} <= set(SPL.columns), f'unexpected columns {list(SPL.columns)}'
sz = SPLIT_CSV.stat().st_size
print(f'{SPLIT_CSV}  ({sz:,} bytes)'
      + ('  <-- matches the 77,012-byte original' if sz == 77012
         else '  <-- NOT 77,012 bytes. Verify this is the split the checkpoint used.'))
print(SPL.split.value_counts().to_string())

raw = pd.read_csv(ANN_CSV)
raw = raw[raw.img_name != 'n0507.mha']          # byte-identical duplicate of n1059
TRAIN = set(SPL[SPL.split == 'train'].img_name)
VAL   = set(SPL[SPL.split == 'val'].img_name)
TEST  = set(SPL[SPL.split == 'test'].img_name) if (SPL.split == 'test').any() else set()
assert not (TRAIN & VAL), 'train and val overlap'
print(f'\ntrain {len(TRAIN)}  val {len(VAL)}  test {len(TEST)}')
print(f'train positives: {raw[(raw.label == 1) & raw.img_name.isin(TRAIN)].img_name.nunique()}')
print(f'val   positives: {raw[(raw.label == 1) & raw.img_name.isin(VAL)].img_name.nunique()}')

json.dump({'source': str(SPLIT_CSV), 'bytes': sz,
           'train': sorted(TRAIN), 'val': sorted(VAL), 'test': sorted(TEST)},
          open(DEST/'split_used.json', 'w'), indent=1)
print(f'\nrecorded -> {DEST}/split_used.json')

## 4 · Stage source images

Bulk copy, because reading `.mha` one at a time over the mount stalls inside a C call
where `KeyboardInterrupt` cannot reach it.

In [ ]:
LOCAL = Path('/content/node21'); LOCAL.mkdir(exist_ok=True)
need = sorted(TRAIN | VAL)
t0, staged = time.time(), 0
for i, n in enumerate(need):
    if not (LOCAL/n).exists():
        subprocess.run(['cp', str(MHA_SRC/n), str(LOCAL/n)], check=True); staged += 1
    if i % 400 == 0: print(f'  {i}/{len(need)}  ({time.time()-t0:.0f}s)')
MHA_DIR = LOCAL
print(f'staged {staged} new, {len(list(LOCAL.glob("*.mha")))} present, '
      f'{sum(f.stat().st_size for f in LOCAL.glob("*.mha"))/1e9:.1f} GB, '
      f'{time.time()-t0:.0f}s')

## 5 · Preprocessing — the upstream one, imported not reimplemented

`node21_detection_baseline.zip` is unzipped and `training_utils` / `algoverse_dataset`
imported directly, so the transforms and the training loop are the same objects the
original run used. Reimplementing them is how drift gets in.

**Real images: native resolution, max-normalised, boxes in native pixels.** No CLAHE, no
percentile clip, no crop, no resize.

**Augmented images: 512 px PNGs, max-normalised, boxes in 512-space.** `NoduleDataset`
reads each image at its own size and takes boxes in that image's pixel space, so mixing a
1024 px `.mha` with a 512 px `.png` is handled correctly by the model's internal transform.

> **Residual mismatch, stated not hidden.** Augmented canvases are 512 px while real images
> are native, so the two training populations differ in resolution. The original run had the
> same property (`.mha` originals plus `copy_paste_outputs` PNGs), so this is consistent with
> Baseline 1's pipeline rather than a new defect — but it is a real confound in the
> augmentation effect and belongs in Limitations. Note also that the paste path here uses
> max-normalise, **not** the CLAHE used to build the RadEdit grid; that is deliberate, so the
> pasted images match the detector's training distribution.

In [ ]:
import cv2, SimpleITK as sitk, sys, zipfile
from PIL import Image

# ---- upstream code, imported rather than reimplemented ----------------------
UP = Path('/content/upstream'); UP.mkdir(exist_ok=True)
_z = None
for _c in [NODE21/'node21_detection_baseline.zip', SOURCE/'node21_detection_baseline.zip']:
    if _c.exists(): _z = _c; break
if _z is not None and not (UP/'node21_detection_baseline').exists():
    with zipfile.ZipFile(_z) as f: f.extractall(UP)
    print(f'unzipped {_z.name}')
_cands = [q.parent for q in UP.rglob('training_utils/transforms.py')]
if _cands:
    sys.path.insert(0, str(_cands[0]))
    import training_utils.transforms as T
    from training_utils.train import train_one_epoch
    UPSTREAM_OK = True
    print(f'imported upstream training_utils from {_cands[0]}')
else:
    UPSTREAM_OK = False
    print('WARNING: upstream training_utils not found. The fallback below is a faithful')
    print('reimplementation of get_transform, but train_one_epoch will not be available.')
    print(f'Put node21_detection_baseline.zip at {NODE21}/ and rerun this cell.')


def load_native(path):
    """Exactly algoverse_dataset.NoduleDataset: squeeze, max-normalise, nothing else."""
    a = np.asarray(sitk.GetArrayFromImage(sitk.ReadImage(str(path)))).squeeze()
    if a.ndim != 2:
        raise ValueError(f'expected 2D after squeeze, got {a.shape} for {path.name}')
    m = float(a.max()) if a.max() > 0 else 1.0
    return a.astype(np.float32)/m


def load_canvas_512(path, size=SIZE):
    """For building pasted images. Max-normalise then resize -- NO CLAHE, so the output
    matches the detector's training distribution rather than the RadEdit grid's."""
    a = load_native(path)
    return np.clip(cv2.resize(a, (size, size), interpolation=cv2.INTER_AREA), 0, 1)


def feather(n, f=0.25):
    y, x = np.mgrid[0:n, 0:n]
    r = np.sqrt((x-(n-1)/2)**2 + (y-(n-1)/2)**2)/((n-1)/2)
    a = np.ones_like(r); e = (r > 1-f) & (r <= 1)
    a[e] = 0.5*(1+np.cos(np.pi*(r[e]-(1-f))/f)); a[r > 1] = 0
    return a


def paste(dest, patch, cx, cy, diam):
    """Intensity matched on the SURROUND, not the core.

    Matching the core sets the nodule to the brightness of the lung it replaces and erases
    the contrast that makes it a nodule -- an earlier version did exactly that and produced
    dark discs. Aligning the surround keeps the lesion's own contrast while removing the
    seam.
    """
    d = int(round(diam))
    px, py = int(round(cx-d/2)), int(round(cy-d/2))
    if px < 0 or py < 0 or px+d > SIZE or py+d > SIZE:
        return None
    patch = cv2.resize(patch, (d, d), interpolation=cv2.INTER_AREA).astype(np.float32)
    alpha = feather(d)
    region = dest[py:py+d, px:px+d]

    core = alpha > 0.9                      # the nodule
    ring = (alpha > 0.05) & (alpha < 0.6)   # surrounding lung
    if ring.sum() < 8:
        ring = ~core
    shift = region[ring].mean() - patch[ring].mean()
    patch = np.clip(patch + shift, 0, 1)

    out = dest.copy()
    out[py:py+d, px:px+d] = alpha*patch + (1-alpha)*region
    return out, (px, py, px+d, py+d)

## 6 · Build the augmented training set

CPU only, and resumable — it skips anything already written. Only **training** chests are
used as canvases; pasting into a validation chest would leak.

In [ ]:
# NOTE: canvases are built with load_canvas_512 (max-normalise + resize, NO CLAHE)
# so pasted images match the detector's training distribution. See section 5.
AUG_CSV = OUT/'augmented.csv'
if AUG_CSV.exists():
    aug = pd.read_csv(AUG_CSV)
    print(f'{len(aug)} augmented images already generated')
else:
    # nodule patches, from TRAINING positives only
    patches = []
    for name, g in raw[raw.label == 1].groupby('img_name'):
        if name not in TRAIN: continue
        img = load_canvas_512(MHA_DIR/name)
        H, W = img.shape
        for r in g.itertuples():
            side = int(max(r.width, r.height) * SIZE / max(W, H) * (W/SIZE))
            side = int(max(r.width, r.height))
            cx, cy = r.x + r.width/2, r.y + r.height/2
            a_nat = load_native(MHA_DIR/name)
            oh, ow = a_nat.shape
            s = min(oh, ow); x0, y0 = (ow-s)//2, (oh-s)//2
            a = load_canvas_512(MHA_DIR/name)
            fx, fy = (cx-x0)/s, (cy-y0)/s
            fs = side/s
            if not (0 < fs < 0.3 and 0.05 < fx < 0.95 and 0.05 < fy < 0.95): continue
            d = int(fs*SIZE)
            if d < 20: continue
            px, py = int(fx*SIZE-d/2), int(fy*SIZE-d/2)
            if px < 0 or py < 0 or px+d > SIZE or py+d > SIZE: continue
            patches.append(a[py:py+d, px:px+d].copy())
    print(f'{len(patches)} nodule patches from training images')
    assert patches, 'no usable patches'

    clean = [n for n in sorted(set(raw[raw.label == 0].img_name) & TRAIN)][:N_AUG_CHESTS]
    rng = np.random.default_rng(SEED)
    rows = []
    for i, name in enumerate(clean):
        stem = name.replace('.mha','')
        bg = load_canvas_512(MHA_DIR/name)
        for k in range(AUG_PER_CHEST):
            iid = f'{stem}_aug{k}'
            if (OUT/'aug'/f'{iid}.png').exists(): continue
            p = patches[rng.integers(len(patches))]
            # anywhere in the middle two-thirds -- crude, but this is an augmentation set,
            # not the controlled grid
            cx, cy = rng.uniform(0.2, 0.8)*SIZE, rng.uniform(0.2, 0.8)*SIZE
            d = p.shape[0]
            comp, box = paste(bg, p, cx, cy, d)
            if comp is None: continue
            Image.fromarray((comp*255).astype(np.uint8)).save(OUT/'aug'/f'{iid}.png')
            rows.append(dict(image_id=iid, src=stem, x0=box[0], y0=box[1],
                             x1=box[2], y1=box[3], patch_px=d))
        if i % 100 == 0: print(f'  {i}/{len(clean)}  {len(rows)} written')
    aug = pd.DataFrame(rows)
    aug.to_csv(AUG_CSV, index=False); shutil.copy(AUG_CSV, DEST)
    print(f'{len(aug)} augmented images')

shutil.make_archive('/content/aug', 'zip', OUT/'aug')
shutil.copy('/content/aug.zip', DEST/'augmented_images.zip')
print('augmented set backed up to Drive')

## 7 · Dataset and model

In [ ]:
import torchvision
from torch.utils.data import DataLoader
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor


def new_model(num_classes=2):
    """train_baseline.py get_model(), verbatim. weights='DEFAULT' is COCO-pretrained --
    the previous version of this notebook used weights=None, i.e. random init, which
    would have made any comparison to Baseline 1 meaningless."""
    m = torchvision.models.detection.fasterrcnn_resnet50_fpn(weights=PRETRAINED)
    inf = m.roi_heads.box_predictor.cls_score.in_features
    m.roi_heads.box_predictor = FastRCNNPredictor(inf, num_classes)
    return m


def get_transform(train):
    """algoverse_dataset.get_transform(), verbatim."""
    if UPSTREAM_OK:
        tf = [T.ToTensor()]
        if train:
            tf.append(T.RandomHorizontalFlip(0.5))
        return T.Compose(tf)
    raise RuntimeError('upstream training_utils.transforms not imported -- see section 5')


def collate(batch):
    return tuple(zip(*batch))


class CXR(torch.utils.data.Dataset):
    """Real NODE21 .mha at native resolution plus pasted 512 px PNGs. Boxes are in each
    image's OWN pixel space, which is what NoduleDataset does and what the model's
    GeneralizedRCNNTransform expects."""
    VALID = {'.mha', '.mhd', '.png', '.jpg', '.jpeg'}

    def __init__(self, real_names, aug_df, gt, transforms):
        self.items = ([('real', n) for n in sorted(real_names)]
                      + [('aug', r.image_id) for r in aug_df.itertuples()])
        self.aug = aug_df.set_index('image_id') if len(aug_df) else aug_df
        self.gt = gt
        self.transforms = transforms

    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        kind, key = self.items[i]
        if kind == 'real':
            a = load_native(MHA_DIR/key)
            boxes = [[r.x, r.y, r.x+r.width, r.y+r.height]
                     for r in self.gt.get(key, pd.DataFrame()).itertuples()]
        else:
            a = np.asarray(Image.open(OUT/'aug'/f'{key}.png').convert('L'),
                           np.float32)/255.0
            r = self.aug.loc[key]
            boxes = [[r.x0, r.y0, r.x1, r.y1]]

        img = Image.fromarray(a, mode='F')
        boxes_t = (torch.as_tensor(boxes, dtype=torch.float32) if boxes
                   else torch.empty([0, 4]))
        area = ((boxes_t[:, 3]-boxes_t[:, 1])*(boxes_t[:, 2]-boxes_t[:, 0])
                if len(boxes) else torch.tensor([0.0]))
        target = {'boxes': boxes_t,
                  'labels': torch.ones(len(boxes), dtype=torch.int64),
                  'image_id': torch.tensor([i]), 'area': area,
                  'iscrowd': torch.zeros(len(boxes), dtype=torch.int64)}
        if self.transforms is not None:
            img, target = self.transforms(img, target)
        if img.dim() == 3 and img.shape[0] == 1:
            img = img.repeat(3, 1, 1)
        return img, target


gt = {n: g for n, g in raw[raw.label == 1].groupby('img_name')}
print(f'{len(gt)} annotated images available')

## 8 · Train — checkpointed every epoch

Each epoch writes to Drive before starting the next. A dead session costs one epoch, not
the run. Re-running this cell resumes from the last checkpoint Drive has.

In [ ]:
# train_baseline.py's loop, with per-epoch Drive checkpointing bolted on.
def latest_ckpt():
    cs = sorted((OUT/'ckpt').glob('epoch_*.pth'), key=lambda q: int(q.stem.split('_')[1]))
    return cs[-1] if cs else None


model = new_model().to(DEV)
params = [q for q in model.parameters() if q.requires_grad]
opt = torch.optim.SGD(params, lr=LR, momentum=MOMENTUM, weight_decay=WEIGHT_DECAY)
sched = torch.optim.lr_scheduler.StepLR(opt, step_size=LR_STEP, gamma=LR_GAMMA)

start = 0
c = latest_ckpt()
if c:
    st = torch.load(c, map_location=DEV, weights_only=False)
    model.load_state_dict(st['model'])
    if 'opt' in st:   opt.load_state_dict(st['opt'])
    if 'sched' in st: sched.load_state_dict(st['sched'])
    start = st['epoch']
    print(f'resuming from {c.name}, epoch {start}')

train_ds = CXR(TRAIN, aug, gt, get_transform(train=True))
print(f'train dataset: {len(train_ds)} items '
      f'({len(TRAIN)} real + {len(aug)} augmented)')
loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True, num_workers=WORKERS,
                    collate_fn=collate)

assert UPSTREAM_OK, ('upstream train_one_epoch is required so the loop matches '
                     'train_baseline.py. See section 5.')

for ep in range(start, EPOCHS):
    t0 = time.time()
    train_one_epoch(model, opt, loader, DEV, ep, print_freq=50)
    sched.step()
    el = time.time()-t0
    ck = OUT/'ckpt'/f'epoch_{ep+1}.pth'
    torch.save({'model': model.state_dict(), 'opt': opt.state_dict(),
                'sched': sched.state_dict(), 'epoch': ep+1,
                'recipe': dict(epochs=EPOCHS, batch=BATCH, lr=LR, seed=SEED,
                               weights=PRETRAINED, momentum=MOMENTUM,
                               weight_decay=WEIGHT_DECAY, lr_step=LR_STEP,
                               lr_gamma=LR_GAMMA, split=str(SPLIT_CSV))},
               ck)
    n = to_drive('ckpt')
    peak = torch.cuda.max_memory_allocated()/1024**2 if torch.cuda.is_available() else 0
    print(f'[epoch {ep+1}] {el/60:.1f} min  peak {peak:.0f} MB  -> {ck.name}  '
          f'(+{n} to Drive)')

print('\ntraining done. the recipe is stored inside every checkpoint.')

## 9 · Evaluate — the same FROC code as Baseline 1

Scoring is resumable and syncs to Drive, for the same reason training is.

In [ ]:
model.eval()

@torch.no_grad()
def detect_native(a):
    """No resize -- the model's own transform scales, as in training. Native-pixel boxes."""
    t = torch.from_numpy(a)[None].repeat(3, 1, 1)
    o = model([t.to(DEV)])[0]
    return o['boxes'].cpu().numpy(), o['scores'].cpu().numpy()


def centre_in_px(b, t):
    cx, cy = (b[0]+b[2])/2, (b[1]+b[3])/2
    return t[0] <= cx <= t[2] and t[1] <= cy <= t[3]


val_names = sorted(VAL)
done = {q.stem for q in (OUT/'dets').glob('*.json')}
todo = [n for n in val_names if n.replace('.mha', '') not in done]
print(f'{len(done)} scored, {len(todo)} to go')

t0 = time.time()
for i, name in enumerate(todo):
    a = load_native(MHA_DIR/name)
    b, sc = detect_native(a)
    json.dump({'boxes': b.tolist(), 'scores': sc.tolist(),
               'shape': [int(a.shape[0]), int(a.shape[1])]},
              open(OUT/'dets'/f'{name.replace(".mha", "")}.json', 'w'))
    if (i+1) % 50 == 0 or i == len(todo)-1:
        print(f'  {i+1}/{len(todo)}  {(time.time()-t0)/60:.1f} min  '
              f'+{to_drive("dets")} to Drive')

rows, nod = [], []
for stem in sorted({q.stem for q in (OUT/'dets').glob('*.json')}):
    d = json.load(open(OUT/'dets'/f'{stem}.json'))
    b, sc = np.array(d['boxes']).reshape(-1, 4), np.array(d['scores'])
    g = gt.get(f'{stem}.mha')
    n_gt = 0
    if g is not None:
        for j, r in enumerate(g.itertuples()):
            t = (r.x, r.y, r.x+r.width, r.y+r.height)
            n_gt += 1
            hits = [ss for bb, ss in zip(b, sc) if centre_in_px(bb, t)]
            nod.append(dict(img_name=stem, nodule=j, x0=t[0], y0=t[1], x1=t[2], y1=t[3],
                            best_score=round(float(max(hits, default=0.0)), 4)))
    rows.append(dict(img_name=stem, n_gt=n_gt, n_boxes=int(len(sc)),
                     max_score=round(float(sc.max()) if len(sc) else 0.0, 4)))

img_df = pd.DataFrame(rows); nod_df = pd.DataFrame(nod)
assert len(img_df) and len(nod_df), 'no val detections -- did scoring run?'
print(f'\n{len(img_df)} val images, {len(nod_df)} val nodules')
img_df.to_csv(OUT/'b3_per_image.csv', index=False); shutil.copy(OUT/'b3_per_image.csv', DEST)
nod_df.to_csv(OUT/'b3_per_nodule.csv', index=False); shutil.copy(OUT/'b3_per_nodule.csv', DEST)

## 10 · FROC and comparison

In [ ]:
def froc(nod_df, img_df, detdir):
    ths = np.unique(np.concatenate([np.linspace(0,1,201), nod_df.best_score.values]))
    cache = []
    for _, r in img_df.iterrows():
        d = json.load(open(detdir/f'{r.img_name}.json'))
        b, sc = np.array(d['boxes']).reshape(-1,4), np.array(d['scores'])
        g = nod_df[nod_df.img_name == r.img_name]
        cache.append((b, sc, [(x.fx0,x.fy0,x.fx1,x.fy1) for x in g.itertuples()]))
    n_img, n_nod, pts = len(img_df), len(nod_df), []
    for t in ths:
        tp = int((nod_df.best_score >= t).sum())
        fp = sum(sum(1 for bb in b[sc>=t] if not any(centre_in_px(bb,g) for g in tg))
                 for b, sc, tg in cache)
        pts.append((t, tp/n_nod, fp/n_img))
    return pd.DataFrame(pts, columns=['threshold','sensitivity','fp_per_image'])

F3 = froc(nod_df, img_df, OUT/'dets')
F3.to_csv(OUT/'b3_froc.csv', index=False); shutil.copy(OUT/'b3_froc.csv', DEST)
OPS = [0.125,0.25,0.5,1,2,4,8]
T3 = pd.DataFrame([dict(fp_per_image=op,
        sensitivity=round(F3[F3.fp_per_image<=op].sensitivity.max(),4)
        if (F3.fp_per_image<=op).any() else np.nan) for op in OPS])
print('Baseline 3 (copy-paste augmented):'); print(T3.to_string(index=False))
print(f'FROC score: {T3.sensitivity.mean():.4f}')

# compare against the NATIVE Baseline 1 pass from notebook 03 section 9 --
# table-froc.csv is the 512+CLAHE pass and is NOT a valid comparator here.
b1 = B1_DIR/'table-preprocessing-comparison.csv'
if b1.exists():
    _c = pd.read_csv(b1)
    _r = _c[(_c.pipeline.str.startswith('native')) & (_c.scope == 'val')]
    assert len(_r), 'no native/val row -- run notebook 03 section 9 first'
    T1 = pd.DataFrame([dict(fp_per_image=o,
                            sensitivity=float(_r.iloc[0][f'sens@{o}']))
                       for o in OPS])
    cmp = T1.merge(T3, on='fp_per_image', suffixes=('_b1','_b3'))
    cmp['delta'] = (cmp.sensitivity_b3 - cmp.sensitivity_b1).round(4)
    print('\nvs Baseline 1:'); print(cmp.to_string(index=False))
    print(f'\nmean FROC: B1 {T1.sensitivity.mean():.4f}  B3 {T3.sensitivity.mean():.4f}  '
          f'delta {T3.sensitivity.mean()-T1.sensitivity.mean():+.4f}')
    print('\nRecipe and split now MATCH Baseline 1 (recovered 2026-09-12), and both')
    print('sides use native preprocessing, so this delta is attributable to the')
    print('augmentation. Two residual confounds to state in Limitations: augmented')
    print('canvases are 512 px while real images are native, and the split is')
    print('image-level because NODE21 ships no patient identifiers.')
else:
    print('\nNative Baseline 1 table not found -- run notebook 03 section 9 first.')

T3.to_csv(OUT/'table-b3-froc.csv', index=False); shutil.copy(OUT/'table-b3-froc.csv', DEST)
print(f'\nsynced: {to_drive("ckpt")} checkpoints, {to_drive("dets")} detections')